# T-ECD: быстрый старт

[T-ECD](https://huggingface.co/datasets/t-tech/T-ECD) — открытый анонимизированный
датасет пользовательских взаимодействий из e-commerce.

В ноутбуке строим простой recommender pipeline на домене **`retail`**:

1. скачиваем данные;
2. делаем time-based split;
3. выбираем top users/items только по train;
4. считаем `HitRate@20`, `NDCG@20`, `Coverage@20`;
5. сравниваем `Random`, `TopPopular` и `iALS`;
6. смотрим sanity-check примеры.

Для retail/FMCG повторные покупки нормальны, поэтому основная постановка:
**next-period recommendation с `recommend_seen=True`**.

## 1. Окружение

In [1]:
!pip install -q polars implicit huggingface_hub pyarrow scipy


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import os

# Лучше выставить до импорта numpy/scipy/implicit.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from pathlib import Path

import numpy as np
import polars as pl
from scipy.sparse import csr_matrix

_ = pl.Config.set_tbl_rows(10)
_ = pl.Config.set_fmt_str_lengths(40)

## 2. Настройки

In [3]:
DATA_DIR = Path("dataset/small")
DOMAIN = "retail"

EVENTS_DIR = DATA_DIR / DOMAIN / "events"
EVENTS_GLOB = str(EVENTS_DIR / "*.pq")

# Время в датасете закодировано как число дней от начала наблюдений.
START_DAY = 1200
N_TRAIN_DAYS = 80
N_VAL_DAYS = 7
N_TEST_DAYS = 7

# Размер подзадачи. Top users/items выбираются только по train.
N_USERS = 10_000
N_ITEMS = 10_000

TOP_N = 50

# Основная retail-постановка: можно рекомендовать уже виденные товары.
MAIN_RECOMMEND_SEEN = True

In [4]:
train_end = START_DAY + N_TRAIN_DAYS
val_end = train_end + N_VAL_DAYS
test_end = val_end + N_TEST_DAYS

print(f"train: ({START_DAY}, {train_end}]")
print(f"val:   ({train_end}, {val_end}]")
print(f"test:  ({val_end}, {test_end}]")

train: (1200, 1280]
val:   (1280, 1287]
test:  (1287, 1294]


## 3. Скачиваем данные

In [5]:
from huggingface_hub import snapshot_download

day_files = [
    f"dataset/small/{DOMAIN}/events/{day:05d}.pq"
    for day in range(START_DAY + 1, test_end + 1)
]

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    allow_patterns=[
        f"dataset/small/{DOMAIN}/items.pq",
        *day_files,
    ],
)

print(f"готово: {len(day_files)} дней событий и каталог товаров")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 95 files:   0%|          | 0/95 [00:00<?, ?it/s]

готово: 94 дней событий и каталог товаров


## 4. Что внутри данных

In [6]:
events_lazy = pl.scan_parquet(EVENTS_GLOB)

print(events_lazy.collect_schema())
events_lazy.head(5).collect()

Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String})


timestamp,user_id,item_id,subdomain,action_type,os
duration[μs],u64,str,str,str,str
1201d 93145µs,34947083,"""fmcg_184807""","""catalog""","""view""","""android"""
1201d 186103µs,17565032,"""fmcg_11453""","""catalog""","""view""","""android"""
1201d 217472µs,17565032,"""fmcg_1096123""","""catalog""","""view""","""android"""
1201d 293276µs,18375942,"""fmcg_121863""","""main""","""view""","""android"""
1201d 293276µs,49994563,"""fmcg_92331""","""catalog""","""view""","""android"""


In [7]:
window_stats = (
    events_lazy
    .filter(
        (pl.col("timestamp").dt.total_days() > START_DAY)
        & (pl.col("timestamp").dt.total_days() <= test_end)
    )
    .select(
        n_events=pl.len(),
        n_users=pl.col("user_id").n_unique(),
        n_items=pl.col("item_id").n_unique(),
        share_non_view=(pl.col("action_type") != "view").mean(),
    )
    .collect()
)

window_stats

n_events,n_users,n_items,share_non_view
u32,u32,u32,f64
171955254,90526,230523,0.043482


In [8]:
ITEMS = (
    pl.scan_parquet(DATA_DIR / DOMAIN / "items.pq")
    .select("item_id", "category", "subcategory")
    .collect()
)

print(f"товаров в каталоге {DOMAIN}: {len(ITEMS):,}")
print(f"с заполненной категорией: {ITEMS['category'].is_not_null().mean():.1%}")

ITEMS.head(5)

товаров в каталоге retail: 250,171
с заполненной категорией: 96.2%


item_id,category,subcategory
str,str,str
"""fmcg_10""","""Office Supplies and Educational Material…","""Stationery Paper"""
"""fmcg_10000""","""Cleaning Supplies and Everyday Household…","""Cleaning and Detergent Products"""
"""fmcg_1000006""",null,null
"""fmcg_1000008""","""Children's Products and Childcare Items""","""Food Products"""
"""fmcg_1000017""","""Foodstuffs and Beverages""","""Sweet Desserts and Confectionery"""


## 5. Time-based split

Делим данные по времени: `train -> validation -> test`.

Top users/items выбираем **только по train**, чтобы не заглядывать в будущее.

In [9]:
def load_events(day_from: int, day_to: int) -> pl.LazyFrame:
    """События за интервал (day_from, day_to] с бинарным label."""
    return (
        pl.scan_parquet(EVENTS_GLOB)
        .filter(
            (pl.col("timestamp").dt.total_days() > day_from)
            & (pl.col("timestamp").dt.total_days() <= day_to)
        )
        .with_columns(
            label=pl.when(pl.col("action_type") == "view")
            .then(0)
            .otherwise(1)
            .cast(pl.Int32)
        )
        .select("timestamp", "user_id", "item_id", "label")
    )


def to_ground_truth(events: pl.DataFrame) -> pl.DataFrame:
    """user_id -> список позитивных item_id."""
    return (
        events
        .filter(pl.col("label") > 0)
        .group_by("user_id")
        .agg(pl.col("item_id").unique())
    )

In [10]:
train_raw = load_events(START_DAY, train_end)

top_users = (
    train_raw
    .filter(pl.col("label") > 0)
    .group_by("user_id")
    .agg(n_pos=pl.len())
    .sort("n_pos", descending=True)
    .head(N_USERS)
    .select("user_id")
)

top_items = (
    train_raw
    .filter(pl.col("label") > 0)
    .group_by("item_id")
    .agg(n_pos=pl.len())
    .sort("n_pos", descending=True)
    .head(N_ITEMS)
    .select("item_id")
)

train_events = (
    train_raw
    .join(top_users, on="user_id")
    .join(top_items, on="item_id")
    .collect()
)

warm_users = (
    train_events
    .filter(pl.col("label") > 0)
    .select("user_id")
    .unique()
)

val_events = (
    load_events(train_end, val_end)
    .join(top_items, on="item_id")
    .collect()
)

test_events = (
    load_events(val_end, test_end)
    .join(top_items, on="item_id")
    .collect()
)

val_gt = to_ground_truth(val_events).join(warm_users, on="user_id")
test_gt = to_ground_truth(test_events).join(warm_users, on="user_id")

CATALOG_SIZE = train_events["item_id"].n_unique()

print(
    f"train: {len(train_events):,} событий, "
    f"{train_events['user_id'].n_unique():,} пользователей, "
    f"{train_events['item_id'].n_unique():,} товаров"
)
print(f"доля позитивов в train: {train_events['label'].mean():.3f}")
print(f"val:  {len(val_gt):,} пользователей с позитивами")
print(f"test: {len(test_gt):,} пользователей с позитивами")
print(f"catalog size for evaluation: {CATALOG_SIZE:,}")

train: 36,846,527 событий, 10,000 пользователей, 10,000 товаров
доля позитивов в train: 0.075
val:  4,491 пользователей с позитивами
test: 4,372 пользователей с позитивами
catalog size for evaluation: 10,000


In [11]:
assert train_events["timestamp"].dt.total_days().max() <= train_end
assert train_events["timestamp"].dt.total_days().min() > START_DAY

print("train закончился на дне:", train_events["timestamp"].dt.total_days().max())
train_events.head(5)

train закончился на дне: 1280


timestamp,user_id,item_id,label
duration[μs],u64,str,i32
1201d 93145µs,34947083,"""fmcg_184807""",0
1201d 781207µs,33381844,"""fmcg_986580""",0
1201d 2s 919054µs,16160205,"""fmcg_544150""",0
1201d 3s 385733µs,9040902,"""fmcg_889448""",0
1201d 3s 816564µs,9040902,"""fmcg_672813""",0


## 6. Метрики

In [12]:
def hit_rate(recs: dict, ground_truth: dict, k: int) -> float:
    scores = [
        float(len(set(recs.get(user, [])[:k]) & set(items)) > 0)
        for user, items in ground_truth.items()
    ]
    return float(np.mean(scores)) if scores else 0.0


def ndcg(recs: dict, ground_truth: dict, k: int) -> float:
    scores = []

    for user, items in ground_truth.items():
        gt = set(items)
        predicted = recs.get(user, [])[:k]

        gains = np.array([1.0 if item in gt else 0.0 for item in predicted])
        discounts = 1.0 / np.log2(np.arange(len(gains)) + 2)
        dcg = float((gains * discounts).sum())

        n_ideal = min(len(gt), k)
        idcg = float((1.0 / np.log2(np.arange(n_ideal) + 2)).sum()) if n_ideal else 0.0

        scores.append(dcg / idcg if idcg > 0 else 0.0)

    return float(np.mean(scores)) if scores else 0.0


def coverage(recs: dict, k: int, catalog_size: int) -> float:
    shown = {item for items in recs.values() for item in items[:k]}
    return len(shown) / catalog_size if catalog_size else 0.0


def evaluate(recs: dict, ground_truth: pl.DataFrame, catalog_size: int, k: int = TOP_N) -> dict:
    gt = dict(ground_truth.iter_rows())

    return {
        f"hit_rate@{k}": hit_rate(recs, gt, k),
        f"ndcg@{k}": ndcg(recs, gt, k),
        f"coverage@{k}": coverage(recs, k, catalog_size),
    }


RESULTS = {}

## 7. A/A и Random

A/A — проверка метрик: рекомендуем пользователю его реальные test-позитивы.
Random — нижняя граница.

In [13]:
rng = np.random.default_rng(42)

aa_recs = {
    user: list(items)[:TOP_N]
    for user, items in test_gt.iter_rows()
}

all_items = (
    train_events
    .filter(pl.col("label") > 0)["item_id"]
    .unique()
    .to_numpy()
)

random_recs = {
    user: list(rng.choice(all_items, size=TOP_N, replace=False))
    for user in test_gt["user_id"].to_list()
}

RESULTS["aa metric check"] = evaluate(aa_recs, test_gt, CATALOG_SIZE)
RESULTS["random"] = evaluate(random_recs, test_gt, CATALOG_SIZE)

print("A/A:    ", RESULTS["aa metric check"])
print("Random: ", RESULTS["random"])

assert RESULTS["aa metric check"][f"hit_rate@{TOP_N}"] > 0.99
assert RESULTS["aa metric check"][f"ndcg@{TOP_N}"] > 0.99

A/A:     {'hit_rate@50': 1.0, 'ndcg@50': 1.0, 'coverage@50': 0.9456}
Random:  {'hit_rate@50': 0.0903476669716377, 'ndcg@50': 0.0034719355889681813, 'coverage@50': 1.0}


## 8. TopPopular

In [14]:
class TopPopular:
    def __init__(
        self,
        last_days: int | None = None,
        recommend_seen: bool = MAIN_RECOMMEND_SEEN,
    ):
        self.last_days = last_days
        self.recommend_seen = recommend_seen
        self.popular: list[str] = []
        self.seen_by_user: dict[int, set[str]] = {}

    def fit(self, events: pl.DataFrame) -> "TopPopular":
        positives = events.filter(pl.col("label") > 0)

        if self.last_days is not None:
            cutoff_day = positives["timestamp"].dt.total_days().max() - self.last_days
            positives_for_pop = positives.filter(pl.col("timestamp").dt.total_days() > cutoff_day)
        else:
            positives_for_pop = positives

        self.popular = (
            positives_for_pop
            .group_by("item_id")
            .agg(n=pl.len())
            .sort("n", descending=True)
            .head(5000)["item_id"]
            .to_list()
        )

        self.seen_by_user = {
            user: set(items)
            for user, items in (
                positives
                .group_by("user_id")
                .agg(pl.col("item_id").unique())
                .iter_rows()
            )
        }

        return self

    def recommend(self, user_ids: list, topn: int = TOP_N) -> dict:
        recs = {}

        for user in user_ids:
            if self.recommend_seen:
                recs[user] = self.popular[:topn]
            else:
                seen = self.seen_by_user.get(user, set())
                recs[user] = [item for item in self.popular if item not in seen][:topn]

        return recs

In [15]:
def tune(candidates: dict, make_model, ground_truth: pl.DataFrame = val_gt) -> tuple[pl.DataFrame, dict, str]:
    users = ground_truth["user_id"].to_list()

    rows = []

    for name, params in candidates.items():
        model = make_model(**params).fit(train_events)
        scores = evaluate(model.recommend(users), ground_truth, CATALOG_SIZE)

        rows.append({"variant": name, **scores})

        print(
            f"  {name}: "
            f"hit_rate={scores[f'hit_rate@{TOP_N}']:.4f}, "
            f"ndcg={scores[f'ndcg@{TOP_N}']:.4f}, "
            f"coverage={scores[f'coverage@{TOP_N}']:.4f}"
        )

    table = pl.DataFrame(rows).sort(f"hit_rate@{TOP_N}", descending=True)
    best_name = table["variant"][0]

    spread = table[f"hit_rate@{TOP_N}"][0] - table[f"hit_rate@{TOP_N}"][-1]
    print(f"  разброс hit_rate: {spread:.4f}")

    return table, candidates[best_name], best_name

In [16]:
print("TopPopular — подбор окна популярности на validation:")

toppop_grid = {
    "all train": {
        "last_days": None,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "last 14 days": {
        "last_days": 14,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "last 7 days": {
        "last_days": 7,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "last 3 days": {
        "last_days": 3,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
}

toppop_table, toppop_best, toppop_best_name = tune(toppop_grid, TopPopular)

print(f"\nлучший TopPopular: {toppop_best_name}")
toppop_table

TopPopular — подбор окна популярности на validation:
  all train: hit_rate=0.7887, ndcg=0.1331, coverage=0.0050
  last 14 days: hit_rate=0.7938, ndcg=0.1351, coverage=0.0050
  last 7 days: hit_rate=0.7974, ndcg=0.1353, coverage=0.0050
  last 3 days: hit_rate=0.7980, ndcg=0.1347, coverage=0.0050
  разброс hit_rate: 0.0094

лучший TopPopular: last 3 days


variant,hit_rate@50,ndcg@50,coverage@50
str,f64,f64,f64
"""last 3 days""",0.798041,0.134686,0.005
"""last 7 days""",0.797373,0.135263,0.005
"""last 14 days""",0.79381,0.135115,0.005
"""all train""",0.788688,0.133053,0.005


In [17]:
toppop = TopPopular(**toppop_best).fit(train_events)
toppop_recs = toppop.recommend(test_gt["user_id"].to_list())

RESULTS["toppop"] = evaluate(toppop_recs, test_gt, CATALOG_SIZE)
RESULTS["toppop"]

{'hit_rate@50': 0.7900274473924978,
 'ndcg@50': 0.1322652590363069,
 'coverage@50': 0.005}

## 9. iALS

In [18]:
from implicit.als import AlternatingLeastSquares


class IALS:
    def __init__(
        self,
        factors: int = 64,
        regularization: float = 0.05,
        alpha: float = 10.0,
        iterations: int = 15,
        weight_by_count: bool = True,
        recommend_seen: bool = MAIN_RECOMMEND_SEEN,
        seed: int = 42,
    ):
        self.alpha = alpha
        self.weight_by_count = weight_by_count
        self.recommend_seen = recommend_seen

        self.model = AlternatingLeastSquares(
            factors=factors,
            regularization=regularization,
            iterations=iterations,
            random_state=seed,
        )

        self.user_map: pl.DataFrame | None = None
        self.item_map: pl.DataFrame | None = None
        self.matrix: csr_matrix | None = None
        self.id2item: list[str] = []
        self.fallback: list[str] = []

    def fit(self, events: pl.DataFrame) -> "IALS":
        positives = events.filter(pl.col("label") > 0)

        users = positives["user_id"].unique().to_list()
        items = positives["item_id"].unique().to_list()

        self.user_map = pl.DataFrame({
            "user_id": users,
            "row": np.arange(len(users), dtype=np.int32),
        })

        self.item_map = pl.DataFrame({
            "item_id": items,
            "col": np.arange(len(items), dtype=np.int32),
        })

        self.id2item = items

        indexed = positives.join(self.user_map, on="user_id").join(self.item_map, on="item_id")

        if self.weight_by_count:
            pairs = indexed.group_by("row", "col").agg(weight=pl.len())

            rows = pairs["row"].to_numpy()
            cols = pairs["col"].to_numpy()
            weights = pairs["weight"].to_numpy().astype(np.float32) * self.alpha
        else:
            rows = indexed["row"].to_numpy()
            cols = indexed["col"].to_numpy()
            weights = np.full(len(indexed), self.alpha, dtype=np.float32)

        self.matrix = csr_matrix(
            (weights, (rows, cols)),
            shape=(len(users), len(items)),
            dtype=np.float32,
        )

        self.model.fit(self.matrix, show_progress=True)

        self.fallback = (
            positives
            .group_by("item_id")
            .agg(n=pl.len())
            .sort("n", descending=True)
            .head(TOP_N)["item_id"]
            .to_list()
        )

        return self

    def recommend(self, user_ids: list, topn: int = TOP_N) -> dict:
        known = self.user_map.filter(pl.col("user_id").is_in(user_ids))
        known_users = known["user_id"].to_list()
        rows = known["row"].to_numpy()

        if len(rows) > 0:
            ids, _ = self.model.recommend(
                rows,
                self.matrix[rows],
                N=topn,
                filter_already_liked_items=not self.recommend_seen,
            )

            recs = {
                user: [self.id2item[i] for i in row_ids]
                for user, row_ids in zip(known_users, ids)
            }
        else:
            recs = {}

        for user in user_ids:
            recs.setdefault(user, self.fallback[:topn])

        return recs

In [19]:
ials_default = IALS().fit(train_events)
ials_default_recs = ials_default.recommend(test_gt["user_id"].to_list())

RESULTS["ials default"] = evaluate(ials_default_recs, test_gt, CATALOG_SIZE)
RESULTS["ials default"]

{'hit_rate@50': 0.8229643183897529,
 'ndcg@50': 0.15338638544331507,
 'coverage@50': 0.5214}

### Короткий tuning iALS на validation

In [20]:
%%time

print("iALS — подбор гиперпараметров на validation:")

ials_grid = {
    "base": {
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "factors=128": {
        "factors": 128,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "alpha=1": {
        "alpha": 1.0,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "alpha=40": {
        "alpha": 40.0,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
    "no count weights": {
        "weight_by_count": False,
        "recommend_seen": MAIN_RECOMMEND_SEEN,
    },
}

ials_table, ials_best, ials_best_name = tune(ials_grid, IALS)

print(f"\nлучший iALS: {ials_best_name}")
ials_table

iALS — подбор гиперпараметров на validation:
  base: hit_rate=0.8372, ndcg=0.1593, coverage=0.5184
  factors=128: hit_rate=0.8577, ndcg=0.1835, coverage=0.7312
  alpha=1: hit_rate=0.8517, ndcg=0.2165, coverage=0.2624
  alpha=40: hit_rate=0.6440, ndcg=0.0663, coverage=0.7986
  no count weights: hit_rate=0.8354, ndcg=0.1602, coverage=0.5223
  разброс hit_rate: 0.2138

лучший iALS: factors=128
CPU times: user 2min 27s, sys: 1min 20s, total: 3min 48s
Wall time: 22 s


variant,hit_rate@50,ndcg@50,coverage@50
str,f64,f64,f64
"""factors=128""",0.857715,0.183472,0.7312
"""alpha=1""",0.851703,0.216495,0.2624
"""base""",0.83723,0.159263,0.5184
"""no count weights""",0.835449,0.160221,0.5223
"""alpha=40""",0.643955,0.066269,0.7986


In [21]:
ials_tuned = IALS(**ials_best).fit(train_events)
ials_tuned_recs = ials_tuned.recommend(test_gt["user_id"].to_list())

RESULTS["ials tuned"] = evaluate(ials_tuned_recs, test_gt, CATALOG_SIZE)
RESULTS["ials tuned"]

{'hit_rate@50': 0.8417200365965233,
 'ndcg@50': 0.17724432236476365,
 'coverage@50': 0.7316}

## 10. Сравнение моделей

In [22]:
comparison = pl.DataFrame([
    {"model": name, **scores}
    for name, scores in RESULTS.items()
])

comparison

model,hit_rate@50,ndcg@50,coverage@50
str,f64,f64,f64
"""aa metric check""",1.0,1.0,0.9456
"""random""",0.090348,0.003472,1.0
"""toppop""",0.790027,0.132265,0.005
"""ials default""",0.822964,0.153386,0.5214
"""ials tuned""",0.84172,0.177244,0.7316


Как читать таблицу:

- `aa metric check` должен давать `HitRate=1` и `NDCG=1`;
- `random` обычно имеет высокий `Coverage`, но низкие `HitRate` и `NDCG`;
- `toppop` — сильный baseline для retail;
- `ials tuned` использует историю пользователя и обычно даёт лучший баланс качества и coverage.

## 11. Sanity-check рекомендаций

Покажем пользователя, для которого iALS попал в test.

Чтобы вывод был компактным, выводим:

- `subcategory_short` — короткую подкатегорию товара;
- `seen_in_train` — был ли товар в train-истории пользователя;
- `hit_in_test` — есть ли товар среди test-позитивов.

In [23]:
def shorten(value: str | None, max_len: int = 34) -> str:
    if value is None:
        return "—"

    value = str(value)
    return value if len(value) <= max_len else value[: max_len - 1] + "…"


def user_train_items(user_id) -> set[str]:
    return set(
        train_events
        .filter((pl.col("user_id") == user_id) & (pl.col("label") > 0))
        ["item_id"]
        .unique()
        .to_list()
    )


def user_test_items(user_id) -> set[str]:
    rows = test_gt.filter(pl.col("user_id") == user_id)

    if rows.height == 0:
        return set()

    return set(rows["item_id"].item().to_list())


def pick_hit_user(recs: dict, gt: pl.DataFrame, topn: int = TOP_N):
    gt_dict = dict(gt.iter_rows())

    for user, true_items in gt_dict.items():
        if set(recs.get(user, [])[:topn]) & set(true_items):
            return user

    return gt["user_id"][0]


def recommendation_table(user_id, recs: dict, topn: int = TOP_N) -> pl.DataFrame:
    train_items = user_train_items(user_id)
    test_items = user_test_items(user_id)
    rec_items = recs[user_id][:topn]

    table = pl.DataFrame({
        "rank": np.arange(1, len(rec_items) + 1, dtype=np.int32),
        "item_id": rec_items,
    })

    return (
        table
        .join(ITEMS, on="item_id", how="left")
        .with_columns(
            subcategory_short=pl.col("subcategory")
            .fill_null("—")
            .map_elements(shorten, return_dtype=pl.String),
            seen_in_train=pl.col("item_id").is_in(list(train_items)),
            hit_in_test=pl.col("item_id").is_in(list(test_items)),
        )
        .select(
            "rank",
            "item_id",
            "subcategory_short",
            "seen_in_train",
            "hit_in_test",
        )
    )


def user_summary(user_id) -> None:
    train_items = user_train_items(user_id)
    test_items = user_test_items(user_id)
    rec_items = set(ials_tuned_recs[user_id][:TOP_N])

    print(f"user_id:          {user_id}")
    print(f"train positives:  {len(train_items):,}")
    print(f"test positives:   {len(test_items):,}")
    print(f"hits@{TOP_N}:         {len(rec_items & test_items):,}")
    print(f"seen recs@{TOP_N}:    {len(rec_items & train_items):,}")


example_user = pick_hit_user(ials_tuned_recs, test_gt, TOP_N)

user_summary(example_user)
recommendation_table(example_user, ials_tuned_recs, TOP_N)

user_id:          33813385
train positives:  94
test positives:   10
hits@50:         4
seen recs@50:    25


rank,item_id,subcategory_short,seen_in_train,hit_in_test
i32,str,str,bool,bool
1,"""fmcg_1047118""","""Grains, Pasta, and Flour Products""",false,false
2,"""fmcg_1171011""","""Personal Hygiene Products""",false,false
3,"""fmcg_335947""","""Fresh Vegetables, Fruits, and Gre…""",false,false
4,"""fmcg_1097902""","""Culinary Additives and Seasonings""",true,false
5,"""fmcg_788467""","""Culinary Additives and Seasonings""",true,false
…,…,…,…,…
46,"""fmcg_711220""","""Dairy Products and Eggs""",true,false
47,"""fmcg_66926""","""—""",false,false
48,"""fmcg_617017""","""Personal Hygiene Products""",false,false


## 12. Похожие товары

У iALS есть item embeddings. Посмотрим соседей популярного товара.
Категории не использовались при обучении, поэтому похожие подкатегории —
полезный sanity-check.

In [24]:
def describe_items_short(item_ids: list[str]) -> pl.DataFrame:
    return (
        pl.DataFrame({
            "rank": np.arange(1, len(item_ids) + 1, dtype=np.int32),
            "item_id": item_ids,
        })
        .join(ITEMS, on="item_id", how="left")
        .with_columns(
            subcategory_short=pl.col("subcategory")
            .fill_null("—")
            .map_elements(shorten, return_dtype=pl.String),
        )
        .select("rank", "item_id", "subcategory_short")
    )


def similar_items(item_id: str, n: int = 10) -> pl.DataFrame:
    col = ials_tuned.item_map.filter(pl.col("item_id") == item_id)["col"].item()
    ids, _ = ials_tuned.model.similar_items(col, N=n + 1)

    neighbours = [
        ials_tuned.id2item[i]
        for i in ids
        if ials_tuned.id2item[i] != item_id
    ][:n]

    return describe_items_short(neighbours)


anchor = (
    train_events
    .filter(pl.col("label") > 0)
    .group_by("item_id")
    .agg(n=pl.len())
    .sort("n", descending=True)
    .join(ITEMS.filter(pl.col("subcategory").is_not_null()), on="item_id")
    .head(1)
)

anchor_id = anchor["item_id"].item()

print("Anchor item:")
display(describe_items_short([anchor_id]))

print("Similar items:")
display(similar_items(anchor_id, n=10))

Anchor item:


rank,item_id,subcategory_short
i32,str,str
1,"""fmcg_1000029""","""Bakery Products"""


Similar items:


rank,item_id,subcategory_short
i32,str,str
1,"""fmcg_26110""","""Bakery Products"""
2,"""fmcg_179451""","""Bakery Products"""
3,"""fmcg_1147754""","""Bakery Products"""
4,"""fmcg_882460""","""Bakery Products"""
5,"""fmcg_406368""","""Bakery Products"""
6,"""fmcg_888400""","""Bakery Products"""
7,"""fmcg_269837""","""Bakery Products"""
8,"""fmcg_1143044""","""Grains, Pasta, and Flour Products"""
9,"""fmcg_365024""","""Cheese Products"""


## 13. Что попробовать дальше

- Расширить tuning iALS: `factors`, `regularization`, `alpha`, `iterations`.
- Добавить веса для разных `action_type`, а не только бинарный label.
- Добавить recency weighting: свежие события важнее старых.
- Сравнить с ItemKNN / UserKNN / SLIM.
- Построить двухэтапную схему: candidate generation + ranking.
- Использовать item/user side information для hybrid и cold-start сценариев.
- Проверить устойчивость результатов на других доменах и больших окнах.

Полезные ссылки:

- датасет: https://huggingface.co/datasets/t-tech/T-ECD
- статья на Habr: https://habr.com/ru/companies/tbank/articles/950696/
- публикация: https://doi.org/10.1145/3770855.3817588